Initalizing bronze layer location and Installing the required libraries

In [0]:
#%pip install requests

In [0]:
#dbutils.library.restartPython()

# Setting Dyanmic Parameters for the rest of code

In [0]:
dbutils.widgets.text("tables", "Patient,Encounter,Observation,Condition")
tables_str = dbutils.widgets.get("tables")

dbutils.widgets.text("raw_layer", "raw")
raw = dbutils.widgets.get("raw_layer")

dbutils.widgets.text("schema_name", "fhir")
schema = dbutils.widgets.get("schema_name")

dbutils.widgets.text("bronze_layer", "bronze")
bronze = dbutils.widgets.get("bronze_layer")

Tables = [t.strip() for t in tables_str.split(",")]

## Importing libraries

In [0]:
import requests
import json
from datetime import datetime, timedelta, UTC
from pyspark.sql.functions import lit, current_timestamp

### Logic for fetcing incremental data via Api call

In [0]:
BASE_URL = "https://hapi.fhir.org/baseR4"

In [0]:
def fetch_fhir_data(Table, last_updated):
    
    base_url = f"{BASE_URL}/{Table}"
    params = f"_lastUpdated=ge{last_updated}&_count=100"
    url = f"{base_url}?{params}"
    
    all_data = []
    api_calls = []   # to track all paginated URLs

    while url:
        api_calls.append(url)   # capture each API call
        
        response = requests.get(url)
        data = response.json()
        
        all_data.append(data)

        # pagination
        next_link = None
        if "link" in data:
            for link in data["link"]:
                if link["relation"] == "next":
                    next_link = link["url"]
        
        url = next_link

    return all_data, base_url, params, api_calls

In [0]:
today = datetime.now(UTC)
dates = [(today - timedelta(days=i)).strftime("%Y-%m-%d") for i in range(3)]

# Raw Layer 
## Loading the data to raw layer 
1. Added additional metadata column extraction_timestamp and api_url_or_params. 

### Notes: 
1. I have used catalogue instead of raw folder as “Both DBFS root and DBFS mounts are deprecated… New accounts are provisioned without access.” this is offical statement from databricks and this is the link of the source  - https://docs.databricks.com/aws/en/dbfs/dbfs-root

2. As by default on cloud the data get stored in delta format. So, not able to store the data as JSON because DBFS has also been deprecated. But in case if you want to save data as a JSON only at an exertnal location you can update raw_path = "External location" .

In [0]:


for Table in Tables:
    
    for date in dates:
        
        data, base_url, params, api_calls = fetch_fhir_data(Table, date)

        df = spark.createDataFrame(
            [(json.dumps(data), json.dumps(api_calls))],
            ["raw_json", "api_url_or_params"]
        )
        
        df = df.withColumn("load_date", lit(date)) \
               .withColumn("extraction_timestamp", current_timestamp()) \
               .withColumn("api_url_or_params",
                    lit(json.dumps({
                            "base_url": base_url,
                            "params": params,
                            "api_calls": api_calls
                            }))
                        )
        
        raw_path = f"{raw}.{schema}.{Table}"
        
        if spark.catalog.tableExists(raw_path):
            df.writeTo(raw_path).append()
            print(raw_path + " table appended")
        else:
            df.writeTo(raw_path).createOrReplace()
            print(raw_path + " table Created")

# Bronze Layer
## Loading the data to Bronze layer

In [0]:
def write_bronze(Table):
    
    raw_path = f"{raw}.{schema}.{Table}"
    
    df = spark.table(raw_path)
    
    bronze_path = f"{bronze}.{schema}.{Table}"

    if spark.catalog.tableExists(bronze_path):
        df.write.format("delta").mode("append").saveAsTable(bronze_path)
        print(bronze_path +" table appended")
    else:
        df.writeTo(bronze_path).createOrReplace()
        print(bronze_path +" table created")

for Table in Tables:
    write_bronze(Table)